# Milan mobile traffic forecasting: full pipeline

Runs the whole project on the real Telecom Italia data, in the same order as `run_all.sh`: ingestion, EDA, time-series analysis, baselines, tuning rounds, final test runs, comparison and report tables.

I run it on Kaggle because my laptop cannot hold the 20.8 GB of raw files. Kaggle gives about 30 GB of RAM and a free GPU for the LSTM rounds.

Add a dataset with the 62 `sms-call-internet-mi-YYYY-MM-DD.txt` files. The next cell searches `/kaggle/input` for those filenames, so the folder name does not matter, but check you have all 62 days: several public copies only cover the first week. On Colab, download them once from https://doi.org/10.7910/DVN/EGZHFV into Drive and point `RAW_DIR` there.

## The problem

Given hourly Internet activity for a grid cell and an origin t, predict t+1, t+6 and t+24 from data up to t. Each horizon gets its own model, so a 24-hour error is not 24 one-hour errors compounded.

The data is Telecom Italia's 100x100 grid over Milan, 10-minute resolution, 1 Nov 2013 to 1 Jan 2014, 20.8 GB. I forecast Internet activity only, 82 % of the total, on 7 cells from the clustering below, never on the citywide aggregate, whose smoothness flatters every model.

Splits are chronological: train to 14 Dec, validation 15 - 21 Dec, test 22 Dec - 1 Jan. The test window covers Christmas on purpose and is untouched until Stage 6.

The metric is MASE against the 24-hour seasonal naive, where below 1 beats it. Scale-free matters because the cells differ in volume by an order of magnitude. The question: does SARIMAX, LightGBM or an LSTM/GRU beat that benchmark, and does the answer change across cells and horizons?

In [ ]:
import os, sys, subprocess, pathlib

ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

def find_raw_dir(root):
    """Return the folder that actually contains sms-call-internet-mi-*.txt files."""
    root = pathlib.Path(root)
    hits = sorted(root.rglob("sms-call-internet-mi-*.txt"))
    if not hits:
        print("nothing matching sms-call-internet-mi-YYYY-MM-DD.txt under", root)
        if root.exists():
            print("what is there:")
            for p in sorted(root.rglob("*"))[:40]:
                print(" ", p)
        return str(root)
    raw_dir = str(hits[0].parent)
    print(f"found {len(hits)} daily files in {raw_dir}")
    return raw_dir

if ON_KAGGLE:
    # don't assume the dataset slug; search whatever you attached under /kaggle/input
    RAW_DIR = find_raw_dir("/kaggle/input")
    WORK = "/kaggle/working"
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAW_DIR = find_raw_dir("/content/drive/MyDrive/milan_raw")
    WORK = "/content"
else:
    RAW_DIR = find_raw_dir("data/raw")
    WORK = "."
print("platform:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local", "| raw dir:", RAW_DIR)

In [ ]:
%cd {WORK}
if not pathlib.Path("Time-Series-Forecasting").exists():
    !git clone -q https://github.com/Samkwizera/Time-Series-Forecasting.git
%cd Time-Series-Forecasting
!pip install -q -r requirements.txt
!pip install -q -e .

The default config looks for the raw files in `data/raw`, which is not where Kaggle puts them. So I copy the config and swap in `RAW_DIR`. Every other path stays relative, so the Parquet intermediates land in the writable working directory.

The next cell counts the 62 daily files locally. It does not call Harvard Dataverse, because that API keeps returning 403 from Kaggle.

In [ ]:
import yaml
cfg = yaml.safe_load(open("config/default.yaml"))
cfg["paths"]["raw_dir"] = RAW_DIR
yaml.safe_dump(cfg, open("config/kaggle.yaml", "w"), sort_keys=False)
os.environ["MILAN_CONFIG"] = "config/kaggle.yaml"

def run(script, *args):
    """Run a pipeline script and stop the notebook if it fails."""
    cmd = [sys.executable, script, "--config", "config/kaggle.yaml", *args]
    print(">", " ".join(cmd))
    subprocess.run(cmd, check=True)

run("scripts/00_download.py", "--verify")

## Stage 1: ingestion

This is the slow part. On the full 20.8 GB it takes about an hour. Wait until it prints the `hourly_*.parquet` paths.

Nothing below works without it. Skip it or stop it early and EDA will go looking for `data/processed/citywide_10min.parquet` and fail.

In [ ]:
run("scripts/01_ingest.py")
from pathlib import Path
import pandas as pd
city = Path("data/processed/citywide_10min.parquet")
assert city.exists(), (
    f"ingest did not write {city}. Scroll up: either the raw files were missing "
    "or the ingest cell was interrupted. Re-run this cell and wait for it to finish."
)
pd.read_csv("reports/tables/memory_log.csv").tail(8)

### What ingestion actually does

Loading all 62 days with default pandas dtypes would take about 17.1 GB, several times the RAM a free session gives you. The memory log above is the measurement rather than the claim.

Each file is scanned lazily and rows are summed over `country_code` *during* the scan. Every (cell, interval) pair appears once per country code, so collapsing that dimension drops roughly 90 % of rows before a frame exists in memory. That is what makes the pipeline feasible at all; downcasting to float32 and uint16 does the rest.

Timestamps become Europe/Rome here, once, so no later stage reasons about time zones. And since the daily files are cut at UTC midnight, the hourly matrix is re-aggregated by timestamp rather than concatenated, which would split the overlapping hours.

Each day adds a median of 16 MB, for a peak of 1,551 MB. One measurement runs against my own design: the citywide 10-minute aggregation peaks at 5,244 MB, because that stage builds a fine-grained frame that aggregating early does not protect.

## Stages 2-3: exploratory and time-series analysis

EDA makes the citywide plots, the spatial maps, and the k-means clustering that picks which cells we forecast (`selected_cells.json`). TSA then runs the stationarity tests, STL/MSTL and ACF/PACF on those cells.

I only show a few figures here. The rest are in `reports/figures/`.

In [ ]:
run("scripts/02_eda.py")
run("scripts/03_tsa.py")
from IPython.display import Image, display
for f in ["eda_citywide_hourly", "eda_daily_profiles", "eda_spatial_internet", "eda_clusters_internet", "tsa_acf_citywide"]:
    display(Image(f"reports/figures/{f}.png", width=900))

### What the figures show

Two cycles dominate, a daily one and a weaker weekly one with lower weekends, and the last week of December departs from both. Weekdays ramp from 6 a.m. and peak in the evening, weekends have no morning ramp, holidays look like Sundays. SMS and calls peak earlier than Internet, which is why I forecast Internet alone rather than pooling the activities.

Activity is very unevenly spread: the busiest 1 % of cells carry 12 % of traffic and the busiest 10 % carry 50 % (Gini 0.62). So evaluation is per cell.

The clustering picks which cells. Silhouette favours k=2; I keep k=4 because it separates two business-shaped from two mixed/suburban profiles, an interpretability choice made against the metric.

The ACF fixes the SARIMA specification: rho(1)=0.954, rho(24)=0.927, rho(168)=0.794, rho(6)=0.120. A seasonal difference at lag 24 gives stationarity under both ADF and KPSS. That fixes most of the choices below: d=0 and D=1, a log1p transform, tree lags of 1-24 plus daily steps to 168, and calendar features for the *target* hour. The near-zero rho(6) predicts h=6 as the horizon where persistence fails.

## Stages 4-5: baselines and tuning rounds

The three naive baselines run first, on both splits, so every later number has something to be compared against.

Each tuning round lives in `experiments/tuning_plan.yaml` with a `why` field I fill in before running it, and every run appends a row to `experiments/experiment_log.md`. The log is the actual trail of what I tried, not a summary written at the end.

SARIMA and LSTM rounds are manual. For LightGBM the manual rounds ablate feature groups first, then 20 Optuna trials handle capacity, where I had no strong prior.

### The three families, and why these three

They span different assumptions about what generates the series, which makes their relative performance interpretable instead of a leaderboard.

**Baselines** - naive, 24-hour and 168-hour seasonal naive. The 24-hour one also scales MASE. They set the bar a learned model must clear to have earned its complexity.

**SARIMAX** - a linear seasonal process, (p,0,q)(P,1,Q) with period 24, per cell. Period 168 would make the state vector far too large, so the weekly cycle is offered as optional Fourier regressors. `dynamic=True` keeps the forecasts genuinely multi-step.

**LightGBM** - one regressor per horizon over engineered lags, trained across cells with a categorical cell id. The L1 objective is the same loss that MAE and MASE measure.

**LSTM / GRU** - a learned representation of the recent sequence, 24 or 168 hours in, all three horizons out.

Tree and network get the same calendar information, so what is compared is inductive bias and not feature access. Spatial CNNs are left out; judging neighbour information fairly needs a different input design.

In [ ]:
run("scripts/04_train.py", "--model", "baselines", "--part", "val")
run("scripts/04_train.py", "--model", "baselines", "--part", "test")
run("scripts/05_tune.py", "--model", "sarima")
run("scripts/05_tune.py", "--model", "lightgbm", "--optuna", "20")
run("scripts/05_tune.py", "--model", "lstm")
print(open("experiments/experiment_log.md").read())

### Reading the log

SARIMAX: MA terms at lags 1 and 24 (s2) cut validation MASE from 0.717 to 0.659, and nothing after improved on it (Fourier 0.666, holiday dummy 0.669, larger specification 0.769). The Fourier terms are the interesting failure, since rho(168)=0.794 is real and they still added nothing.

LightGBM: target-hour calendar features gave the best run of the study (g3, 0.604), as the EDA predicted. Rolling statistics made it worse (0.664). The best of 20 Optuna trials reached 0.633 but held `use_rolling=True` fixed, so it shows no capacity setting rescues the rolling variant, not that g3 is optimal.

Recurrent: a week of context did not help (l2 0.747 against l1 0.721) until the network was widened (l3, 0.714), and the GRU reached 0.711. Those two are 0.003 apart on one seed, smaller than initialisation noise; I took the GRU for having fewer parameters, not for being better.

Several well-motivated changes failed to generalise even to validation. That is the first hint of what follows.

## Stage 6: final test runs and comparison

For each model I take the config with the lowest validation MASE and run it once on the test split, 22 Dec to 1 Jan. That window covers the holidays on purpose. The test period is not touched before this point.

`06_compare.py` then scores everything against the baselines, runs the Diebold-Mariano tests and writes the failure-analysis figures.

In [ ]:
# same selection rule as run_all.sh, written out here so the chosen runs are visible in the output
import json, glob
def best(model):
    runs = [json.load(open(p)) for p in glob.glob(f"experiments/runs/{model}/*_val.json")]
    r = min(runs, key=lambda r: r["metrics"]["mase"])
    return r["run_id"].removesuffix("_val"), r["params"]
runs = {m: best(m) for m in ["sarima", "lightgbm", "lstm"]}
for m, (rid, params) in runs.items():
    print(m, rid, {k: v for k, v in params.items() if k in ("order","seasonal_order","fourier_k","lags","num_leaves","learning_rate","cell","hidden_size","num_layers","input_window")})
for m, (rid, params) in runs.items():
    cell = params.get("cell", m) if m == "lstm" else m
    run("scripts/04_train.py", "--model", cell, "--part", "test", "--run", rid,
        "--params", json.dumps(params), "--note", "final test run of best validation config")
lstm_cell = runs["lstm"][1].get("cell", "lstm")
run("scripts/06_compare.py", "--runs", f"sarima={runs['sarima'][0]}",
    f"lightgbm={runs['lightgbm'][0]}", f"{lstm_cell}={runs['lstm'][0]}")
pd.read_csv("reports/tables/results_summary.csv")

In [ ]:
for f in ["results_mase_by_horizon_and_cell", "results_forecasts_h1", "failure_daily_error", "failure_hour_daytype"]:
    display(Image(f"reports/figures/{f}.png", width=900))

### What the comparison shows

The 24-hour seasonal naive is strongest overall at MASE 0.863. SARIMAX (1.427) and LightGBM (1.432) are both above 1, and the GRU is far behind at 2.223. Diebold-Mariano agrees: against the seasonal naive, SARIMAX is better in 7 of 21 comparisons and worse in 10, LightGBM better in 2 and worse in 14, the GRU better in 2 and worse in 15.

The horizon separates the methods more sharply than the model family does. SARIMAX is the best method in the study at one hour (0.399) and degrades to 2.341 at 24; LightGBM degrades less and overtakes it there (1.568). The two differ by 0.005 overall, far less than the spread across cells, so I do not rank them.

Daily error peaks on 25-26 December and is highest on holidays for every learned model. The test window was chosen to cover Christmas, so this is the stress test working as intended.

One caveat on all of it: a single test window, seven cells and one seed. The DM tests are the only uncertainty estimate here.

## Stage 7: report assets

No number in the report is typed by hand. `07_report_assets.py` turns the CSVs into LaTeX tables and `\newcommand` macros under `report/generated/`.

The last cell zips `reports/`, `experiments/` and those generated files. Kaggle has no LaTeX, so I unpack them in my local clone and build the PDF there.

In [ ]:
run("scripts/07_report_assets.py")
!zip -qr results_bundle.zip reports experiments report/generated
print("download results_bundle.zip, unpack it in the local clone, then run report/build.sh")

## Conclusion

The 24-hour seasonal naive is best overall at MASE 0.863, and no learned model beats it beyond one hour ahead. That is a real negative result rather than a bug: a holiday fortnight is mainly a shift in level, and re-anchoring on a recent observation absorbs it without estimation, which is why the baseline's error barely grows with horizon while SARIMAX degrades from 0.399 at h=1 to 2.341 at h=24.

SARIMAX and LightGBM are indistinguishable in aggregate and better read by horizon. The GRU does not justify its complexity on six weeks of data. Ranking varies with horizon but not demonstrably with cell profile, so that half of the research question gets no positive claim.

What I would do next, in order. Forecast the residual from the last observed same-hour value, which gives the models the benchmark's own advantage. Train on all 10,000 cells with cell-type embeddings, which would give the sequence model the training volume it lacked. And use a longer dataset covering several holiday periods, since that is the only way to estimate holiday effects instead of flagging them.

Limits to hold in mind when reading any of the above: seven cells, one activity, one seed, a single validation week that contains no holiday, and a deliberately hard test period.